In [1]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

# =========================
# 1. CẤU HÌNH FILE
# =========================
INPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/12_skill_master.xlsx"
OUTPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/13_skill_mapping_seed.xlsx"


# =========================
# 2. HÀM ĐỌC FILE
# =========================
def read_table(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Định dạng file chưa hỗ trợ: {path.suffix}")


# =========================
# 3. CHUẨN HÓA TEXT
# =========================
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9+#./\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# =========================
# 4. HÀM TÌM CỘT
# =========================
def find_column(df: pd.DataFrame, candidates: list[str], required=True):
    cols_map = {normalize_text(c): c for c in df.columns}

    for cand in candidates:
        cand_norm = normalize_text(cand)
        if cand_norm in cols_map:
            return cols_map[cand_norm]

    for col in df.columns:
        col_norm = normalize_text(col)
        for cand in candidates:
            cand_norm = normalize_text(cand)
            if cand_norm in col_norm or col_norm in cand_norm:
                return col

    if required:
        raise KeyError(
            f"Không tìm thấy cột phù hợp. Candidates={candidates}. "
            f"Các cột hiện có: {list(df.columns)}"
        )
    return None


# =========================
# 5. RULE GÁN SKILL
# =========================
RULES = [
    # Frontend frameworks / libraries
    {
        "keywords": [
            "react", "reactjs", "react.js", "next", "nextjs", "next.js",
            "vue", "vuejs", "vue.js", "nuxt", "nuxtjs", "nuxt.js",
            "angular", "angularjs", "svelte", "jquery", "redux",
            "tailwind", "bootstrap", "material ui", "chakra ui"
        ],
        "skill_group": "Framework / Library",
        "skill_subgroup": "Frontend Framework",
        "mapped_taxonomy_group": "Software Development",
        "mapped_taxonomy_subgroup": "Frontend"
    },

    # Frontend core web
    {
        "keywords": [
            "html", "css", "sass", "scss", "javascript", "typescript", "ajax", "dom"
        ],
        "skill_group": "Programming Language",
        "skill_subgroup": "Web Development",
        "mapped_taxonomy_group": "Software Development",
        "mapped_taxonomy_subgroup": "Frontend"
    },

    # Backend languages
    {
        "keywords": [
            "java", "c#", ".net", "dotnet", "php", "ruby", "go", "golang",
            "node.js", "nodejs", "express", "nestjs", "spring", "spring boot",
            "laravel", "django", "flask", "fastapi", "asp.net", "aspnet",
            "servlet", "hibernate"
        ],
        "skill_group": "Programming Language / Framework",
        "skill_subgroup": "Backend Development",
        "mapped_taxonomy_group": "Software Development",
        "mapped_taxonomy_subgroup": "Backend"
    },

    # Mobile
    {
        "keywords": [
            "android", "ios", "swift", "kotlin", "flutter", "react native",
            "xamarin", "objective-c", "objective c"
        ],
        "skill_group": "Mobile Development",
        "skill_subgroup": "Mobile App Development",
        "mapped_taxonomy_group": "Software Development",
        "mapped_taxonomy_subgroup": "Mobile"
    },

    # Game
    {
        "keywords": [
            "unity", "unreal", "unreal engine", "godot", "cocos", "game engine"
        ],
        "skill_group": "Game Development",
        "skill_subgroup": "Game Engine / Tool",
        "mapped_taxonomy_group": "Software Development",
        "mapped_taxonomy_subgroup": "Game Development"
    },

    # Databases
    {
        "keywords": [
            "mysql", "postgresql", "postgres", "sql server", "mssql", "oracle",
            "mongodb", "redis", "sqlite", "mariadb", "cassandra", "elasticsearch"
        ],
        "skill_group": "Database",
        "skill_subgroup": "Database Management",
        "mapped_taxonomy_group": "Software Development",
        "mapped_taxonomy_subgroup": "Backend"
    },

    # Data analysis / BI
    {
        "keywords": [
            "power bi", "tableau", "bi", "business intelligence", "data visualization",
            "excel", "statistics", "statistical analysis", "reporting"
        ],
        "skill_group": "Data / BI Tool",
        "skill_subgroup": "Data Analysis",
        "mapped_taxonomy_group": "Data & AI",
        "mapped_taxonomy_subgroup": "Data Analysis"
    },

    # Data engineering
    {
        "keywords": [
            "etl", "data warehouse", "spark", "hadoop", "airflow", "kafka",
            "databricks", "bigquery", "snowflake"
        ],
        "skill_group": "Data Engineering Tool",
        "skill_subgroup": "Data Engineering",
        "mapped_taxonomy_group": "Data & AI",
        "mapped_taxonomy_subgroup": "Data Engineering"
    },

    # AI / ML
    {
        "keywords": [
            "python", "pandas", "numpy", "scikit", "scikit-learn", "sklearn",
            "tensorflow", "pytorch", "keras", "machine learning", "deep learning",
            "nlp", "computer vision", "llm", "langchain", "llamaindex", "hugging face",
            "huggingface", "opencv", "xgboost"
        ],
        "skill_group": "AI / Data Tool",
        "skill_subgroup": "Machine Learning / AI",
        "mapped_taxonomy_group": "Data & AI",
        "mapped_taxonomy_subgroup": "AI / Machine Learning"
    },

    # DevOps / Cloud
    {
        "keywords": [
            "docker", "kubernetes", "k8s", "jenkins", "gitlab ci", "github actions",
            "terraform", "ansible", "helm", "prometheus", "grafana", "argo cd",
            "ci/cd", "cicd"
        ],
        "skill_group": "Cloud / DevOps Tool",
        "skill_subgroup": "DevOps / Automation",
        "mapped_taxonomy_group": "Infrastructure & Cloud",
        "mapped_taxonomy_subgroup": "DevOps / SRE"
    },

    # Cloud platforms
    {
        "keywords": [
            "aws", "amazon web services", "azure", "gcp", "google cloud",
            "cloud computing", "cloud architecture"
        ],
        "skill_group": "Cloud Platform",
        "skill_subgroup": "Cloud Services",
        "mapped_taxonomy_group": "Infrastructure & Cloud",
        "mapped_taxonomy_subgroup": "Cloud"
    },

    # System / network
    {
        "keywords": [
            "linux", "windows server", "unix", "networking", "tcp/ip", "dns",
            "dhcp", "firewall", "vpn", "routing", "switching", "system administration"
        ],
        "skill_group": "System / Network",
        "skill_subgroup": "System / Network Administration",
        "mapped_taxonomy_group": "Infrastructure & Cloud",
        "mapped_taxonomy_subgroup": "System / Network"
    },

    # Security
    {
        "keywords": [
            "cybersecurity", "information security", "penetration testing",
            "penetration test", "pentest", "owasp", "siem", "soc", "iso 27001",
            "vulnerability assessment", "ethical hacking"
        ],
        "skill_group": "Security Tool / Practice",
        "skill_subgroup": "Cybersecurity",
        "mapped_taxonomy_group": "Security",
        "mapped_taxonomy_subgroup": "Cybersecurity"
    },

    # QA / Testing
    {
        "keywords": [
            "selenium", "cypress", "playwright", "postman", "jmeter",
            "test automation", "manual testing", "qa", "quality assurance", "api testing"
        ],
        "skill_group": "Testing Tool",
        "skill_subgroup": "Testing / QA",
        "mapped_taxonomy_group": "QA & Testing",
        "mapped_taxonomy_subgroup": "Testing / QA"
    },

    # BA / Product / PM
    {
        "keywords": [
            "business analysis", "business analyst", "requirement gathering",
            "uml", "user story", "scrum", "agile", "jira", "confluence",
            "product management", "project management", "kanban"
        ],
        "skill_group": "Business / Product / Delivery",
        "skill_subgroup": "Business Analysis / Project Delivery",
        "mapped_taxonomy_group": "Product / Business / Delivery",
        "mapped_taxonomy_subgroup": "Business Analysis"
    },

    # Design
    {
        "keywords": [
            "figma", "adobe xd", "sketch", "wireframing", "prototyping",
            "ui design", "ux design", "interaction design"
        ],
        "skill_group": "Design Tool",
        "skill_subgroup": "UI/UX Design",
        "mapped_taxonomy_group": "Design",
        "mapped_taxonomy_subgroup": "UI/UX Design"
    },

    # Support / operations
    {
        "keywords": [
            "helpdesk", "technical support", "service desk", "it support",
            "troubleshooting", "incident management"
        ],
        "skill_group": "IT Support",
        "skill_subgroup": "Support / Operations",
        "mapped_taxonomy_group": "Support & Operations",
        "mapped_taxonomy_subgroup": "Support / Operations"
    }
]


def assign_skill_mapping(skill_name: str):
    name = normalize_text(skill_name)

    for rule in RULES:
        for kw in rule["keywords"]:
            kw_norm = normalize_text(kw)
            if kw_norm in name:
                return pd.Series([
                    rule["skill_group"],
                    rule["skill_subgroup"],
                    rule["mapped_taxonomy_group"],
                    rule["mapped_taxonomy_subgroup"]
                ])

    return pd.Series([
        "Other Skill",
        "Unclassified",
        "Other IT",
        "Unclassified"
    ])


# =========================
# 6. CHẠY CHÍNH
# =========================
def main():
    df = read_table(INPUT_FILE)
    print("Đã đọc file:", df.shape)
    print("Các cột hiện có:", df.columns.tolist())

    skill_name_col = find_column(df, ["skill_name", "preferredLabel", "skillLabel", "label", "name"])

    rename_map = {
        skill_name_col: "skill_name"
    }

    # đổi tên nếu cột đã có
    optional_cols = {
        "skill_id": ["skill_id"],
        "skill_type": ["skill_type"],
        "group": ["group"],
        "relation_count": ["relation_count"],
        "notes": ["notes"]
    }

    for target_col, candidates in optional_cols.items():
        found_col = find_column(df, candidates, required=False)
        if found_col:
            rename_map[found_col] = target_col

    df = df.rename(columns=rename_map).copy()

    # thêm cột nếu thiếu
    required_output_cols = [
        "skill_id", "skill_name", "skill_type", "group", "relation_count", "notes"
    ]
    for col in required_output_cols:
        if col not in df.columns:
            df[col] = None

    # gán mapping
    df[[
        "skill_group",
        "skill_subgroup",
        "mapped_taxonomy_group",
        "mapped_taxonomy_subgroup"
    ]] = df["skill_name"].apply(assign_skill_mapping)

    # sắp xếp
    df = df.sort_values(
        by=["mapped_taxonomy_group", "mapped_taxonomy_subgroup", "skill_name"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    # sắp xếp cột đầu ra
    output_cols = [
        "skill_id",
        "skill_name",
        "skill_type",
        "group",
        "relation_count",
        "skill_group",
        "skill_subgroup",
        "mapped_taxonomy_group",
        "mapped_taxonomy_subgroup",
        "notes"
    ]
    output_cols = [c for c in output_cols if c in df.columns]
    df_final = df[output_cols]

    # xuất file
    output_path = Path(OUTPUT_FILE)
    if output_path.suffix.lower() in [".xlsx", ".xls"]:
        df_final.to_excel(output_path, index=False)
    else:
        df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Đã tạo file: {OUTPUT_FILE}")
    print("Tổng số skill:", len(df_final))
    print(df_final.head(15))


if __name__ == "__main__":
    main()

Đã đọc file: (1172, 10)
Các cột hiện có: ['skill_id', 'skill_name', 'skill_type', 'group', 'relation_count', 'skill_group', 'skill_subgroup', 'mapped_taxonomy_group', 'mapped_taxonomy_subgroup', 'notes']
Đã tạo file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/13_skill_mapping_seed.xlsx
Tổng số skill: 1172
    skill_id                                 skill_name skill_type     group  \
0        NaN              Python (computer programming)   optional  extended   
1        NaN                            computer vision   optional      core   
2        NaN                              deep learning   optional      core   
3        NaN                           machine learning  essential      core   
4        NaN                   utilise machine learning  essential      core   
5        NaN  Frostbite (digital game creation systems)  essential      core   
6        NaN                ICT accessibility stan

### Kết quả bước tạo skill mapping ban đầu
Từ file `12_skill_master.xlsx`, hệ thống đã gán nhóm kỹ năng và nhóm nghề ánh xạ cho từng skill bằng rule từ khóa, tạo ra file 

`13_skill_mapping_seed.xlsx`. Đây là bản phân loại ban đầu, trong đó mỗi skill được gán vào `skill_group`, `skill_subgroup`, 

`mapped_taxonomy_group` và `mapped_taxonomy_subgroup`. Sau bước này, cần tiếp tục rà soát và chỉnh sửa thủ công các skill chưa 

được phân loại đúng hoặc còn thuộc nhóm `Other Skill / Unclassified`.